# 02 - Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/raw/diabetes.csv")
FIG_PATH = Path("../outputs/figures")
FIG_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

zero_check_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df_eda = df.copy()
df_eda[zero_check_cols] = df_eda[zero_check_cols].replace(0, np.nan)


**Feature distributions by outcome**

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(df.drop(columns="Outcome").columns):
    sns.histplot(data=df_eda, x=col, hue="Outcome", kde=True, ax=axes[i], bins=25)
    axes[i].set_title(col)

axes[-1].axis("off")
plt.tight_layout()
plt.savefig(FIG_PATH / "feature_distributions.png", dpi=120)
plt.show()


**Correlation heatmap**

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df_eda.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(FIG_PATH / "correlation_heatmap.png", dpi=120)
plt.show()


**Boxplots — spotting outliers per feature**

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(df.drop(columns="Outcome").columns):
    sns.boxplot(data=df_eda, y=col, ax=axes[i], color="skyblue")
    axes[i].set_title(col)

plt.tight_layout()
plt.savefig(FIG_PATH / "boxplots.png", dpi=120)
plt.show()


**Class imbalance visualization**

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Outcome", palette=["#4C72B0", "#DD8452"])
plt.title(f"Class Balance (0={sum(df.Outcome==0)}, 1={sum(df.Outcome==1)})")
plt.xlabel("Outcome (0 = Non-diabetic, 1 = Diabetic)")
plt.tight_layout()
plt.savefig(FIG_PATH / "class_balance.png", dpi=120)
plt.show()

print(f"Positive class ratio: {df['Outcome'].mean():.2%}")


**Pairwise relationships for the strongest predictors**

Glucose, BMI, and Age are typically the strongest signal features for this dataset — worth a closer pairwise look.

In [ ]:
subset = df_eda[["Glucose", "BMI", "Age", "Outcome"]]
sns.pairplot(subset, hue="Outcome", palette=["#4C72B0", "#DD8452"], diag_kind="kde")
plt.savefig(FIG_PATH / "pairplot_key_features.png", dpi=120)
plt.show()


**EDA takeaways**
- Glucose shows the clearest separation between classes — expected, as it's the most direct clinical marker.
- Insulin and SkinThickness have the most missing values (disguised as 0) and the widest spread/outliers — imputation strategy in preprocessing matters most for these two.
- Class imbalance (65% / 35%) is moderate, not extreme, but still enough to bias plain accuracy — worth carrying `class_weight` or resampling into modeling.
- BMI and Age show mild-to-moderate correlation with Outcome individually but stronger signal in combination (see Glucose×BMI interaction considered later).